## 🏗️ LeetCode 981: Time Based Key-Value Store
---

<div style="padding: 15px; border-left: 8px solid #9C27B0; background-color: #f3e5f5; color: #4a148c; border-radius: 4px;">
    <strong>The Core Insight:</strong> A regular dictionary gives us O(1) key lookups. If we store a LIST of <code>(timestamp, value)</code> pairs for each key, and we know timestamps are strictly strictly increasing (append-only), that list is naturally sorted. This means we can use Binary Search instead of scanning the list backward.
</div>

### 🛠️ The Mathematical Model

For a single key with $k$ historical values, finding the floor of a given timestamp $T$ ($t \le T$):
- **Linear search (backward):** $O(k)$ worst case
- **Binary Search (bisect_right):** $O(\log k)$ worst case

If we have $N$ total `set` operations over $M$ keys, average list length is $N/M$. The time to retrieve is reduced from $T \propto N/M$ to $T \propto \log(N/M)$.

---

### 📋 Problem

Design a time-based key-value data structure that can store multiple values for the same key at different time stamps and retrieve the key's value at a certain timestamp.

Implement the `TimeMap` class:
- `TimeMap()` Initializes the object of the data structure.
- `void set(String key, String value, int timestamp)` Stores the key `key` with the value `value` at the given time `timestamp`.
- `String get(String key, int timestamp)` Returns a value such that `set` was called previously, with `timestamp_prev <= timestamp`. If there are multiple such values, it returns the value associated with the largest `timestamp_prev`. If there are no values, it returns `""`.

**Note:** All timestamps `timestamp` of `set` are strictly increasing.

**Example 1:**
```
Input
["TimeMap", "set", "get", "get", "set", "get", "get"]
[[], ["foo", "bar", 1], ["foo", 1], ["foo", 3], ["foo", "bar2", 4], ["foo", 4], ["foo", 5]]
Output
[null, null, "bar", "bar", null, "bar2", "bar2"]
```

**Constraints:** 1 <= key.length, value.length <= 100 | 1 <= timestamp <= 10^7 | All timestamps in `set` are strictly increasing

### 🧠 Choose Your Mental Model

<table style="width:100%; border-collapse: collapse;">
    <tr style="background-color: #f2f2f2; text-align: left;">
        <th style="padding: 12px; border: 1px solid #ddd;">Model</th>
        <th style="padding: 12px; border: 1px solid #ddd;">The "Story"</th>
        <th style="padding: 12px; border: 1px solid #ddd;">Mechanism</th>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Dictionary of Lists</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"The master index is a dict pointing to a timeline. Because events only happen forwards in time, every timeline is automatically sorted. I binary search the timeline."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Dict lookup O(1) + Binary search on list O(log K)</td>
    </tr>
    <tr>
        <td style="padding: 12px; border: 1px solid #ddd;"><b>Git History</b></td>
        <td style="padding: 12px; border: 1px solid #ddd;">"Each key is a file, the list is its commit history. Finding a value at time T means finding the commit immediately before or exactly at T."</td>
        <td style="padding: 12px; border: 1px solid #ddd;">Append-only data structure guarantees monotonic ordering property</td>
    </tr>
</table>

---

### ⚠️ Performance Warning

<div style="padding: 10px; border: 1px solid #ffe58f; background-color: #fffbe6; border-radius: 4px;">
    <strong>Note:</strong> In Python, we can't easily iterate a list backwards without slicing or indices. Scanning from the end sequentially is $O(k)$ where $k$ is the number of writes for that key. If a key is written 100,000 times, every query scans.
</div>

## 🐢 Approach 1: Brute Force — Search Backward Stringently $O(k)$

In [ ]:
from collections import defaultdict

class TimeMapBrute:
    def __init__(self):
        # dict mapping key -> list of [timestamp, value]
        self.store = defaultdict(list)

    def set(self, key: str, value: str, timestamp: int) -> None:
        self.store[key].append([timestamp, value])

    def get(self, key: str, timestamp: int) -> str:
        # Get the history for this key
        history = self.store.get(key, [])
        if not history:
            return ""
        
        # Iterate backwards through the history list
        for i in range(len(history) - 1, -1, -1):
            if history[i][0] <= timestamp:
                return history[i][1]
        
        return ""


# Test Cases
tm = TimeMapBrute()
tm.set("foo", "bar", 1)
print(tm.get("foo", 1))  # Expected: "bar"
print(tm.get("foo", 3))  # Expected: "bar"
tm.set("foo", "bar2", 4)
print(tm.get("foo", 4))  # Expected: "bar2"
print(tm.get("foo", 5))  # Expected: "bar2"

## 🔬 Complexity Analysis: $O(k)$ vs. $O(\log k)$

<div style="padding: 15px; border-left: 8px solid #2196F3; background-color: #f0f7ff; color: #0d47a1; border-radius: 4px;">
    <strong>The Core Insight:</strong> The problem guarantees that `set` calls use strictly increasing timestamps. This means `store[key]` is ALWAYS ascending sorted by timestamp. Anytime we search a sorted list, we can drop from $O(n)$ to $O(\log n)$.
</div>

---

## 📉 Why Brute Force Fails: The $O(k)$ Trap

The worst-case scenario is searching for a timestamp earlier than ANY stored timestamp for that key. We have to scan the entire historical list just to return `""`.

| Operation | Brute Force Performance | Reason |
|------------|------------------------|--------|
| **set()** | O(1)$ | Simple append |
| **get()** | O($k$) where $k$ is history size | Iterating backwards to find the first $\le$ match |

---

## 🚀 The Optimal Approach: Binary Search using `bisect` — $O(\log k)$

We need to find the rightmost timestamp that is $\le$ the query timestamp. Python's `bisect_right` finds the insertion point AFTER any equal elements.

If we pass a tuple `(timestamp, chr(127))` to `bisect_right`:
- It compares the first tuple element (timestamp) first.
- If timestamps match, it compares the second element.
- `chr(127)` is greater than any standard ASCII character, ensuring the search pushes the insertion index to the RIGHT of any identical timestamp.
- We then check `idx - 1`. If `idx == 0`, no earlier timestamps exist.

---

## ✅ Mathematical Proof

$$T_{\text{get}}(k) = O(\log k)$$
For $10^5$ history entries:
Linear: ~100,000 steps
Binary: ~17 steps.

<div style="padding: 10px; border-left: 8px solid #4CAF50; background-color: #f1f8f4; color: #1b5e20; border-radius: 4px;">
    <strong>✅ Summary:</strong> Because writes are chronologically ordered, the timeline is pre-sorted. Floor queries (greatest $\le X$) are trivially solved with binary search.
</div>

## 🚀 Approach 2: Binary Search (Bisect Right) — $O(\log k)$
---

Use Python's `bisect` library for robust, C-optimized binary search.


In [ ]:
from collections import defaultdict
import bisect

class TimeMap:
    def __init__(self):
        # Store key -> list of (timestamp, value)
        self.store = defaultdict(list)

    def set(self, key: str, value: str, timestamp: int) -> None:
        # O(1) append. List remains sorted by timestamp because of problem constraints
        self.store[key].append((timestamp, value))

    def get(self, key: str, timestamp: int) -> str:
        # O(1) dict lookup
        history = self.store.get(key, [])
        if not history:
            return ""
        
        # Use bisect_right to find the insertion point.
        # We search with a dummy value chr(127) which is > all valid ASCII chars.
        # This guarantees we insert after an exact timestamp match.
        idx = bisect.bisect_right(history, (timestamp, chr(127)))
        
        # If idx == 0, it means the required timestamp is SMALLER than all recorded ones
        if idx == 0:
            return ""
        
        # Return the value immediately left of the insertion point
        return history[idx - 1][1]


tm = TimeMap()
tm.set("foo", "bar", 1)
tm.set("foo", "bar2", 4)
print("Optimal:", tm.get("foo", 4))   # Expected: "bar2"
print("Optimal:", tm.get("foo", 3))   # Expected: "bar"
print("Optimal:", tm.get("foo", 5))   # Expected: "bar2"
print("Optimal:", tm.get("foo", 0))   # Expected: "" (before first timestamp)

## 🎤 5 Interview Q&A

### Q1: Why is this highly relevant to data engineering?

**Answer:** Time-series lookups are everywhere in data systems — "give me the metric value at time T" or "what was the most recent configuration before this deployment?" Monitoring systems (APM tools), SLA calculations, capacity trending, audit trails — all require floor queries on timestamps. This problem is the algorithmic core of every such lookup.

---

### Q2: What is `bisect.bisect_right` and how does it work?

**Answer:** `bisect_right(array, value)` returns the rightmost index where `value` could be inserted to keep the array sorted. All elements at `[0, result-1]` are ≤ `value`. Elements at `[result, end]` are > `value`. We use `bisect_right` rather than `bisect_left` because we want all timestamps equal to the query to count as valid (≤ not <).

---

### Q3: What is the time complexity of the optimal approach and why?

**Answer:** `set()` = O(1) — just an append. `get()` = O(log n) where n is the number of `set()` calls for that key — binary search on the sorted timestamp list. Overall for k operations: O(k) for set calls, O(k log k) worst case for get calls.

---

### Q4: What if timestamps in set() were NOT guaranteed to be increasing?

**Answer:** We'd need to sort the list on each `set()` call — O(n log n) per insert, making it worse than brute force for frequent writes. Or use a sorted container like `SortedList` from the `sortedcontainers` library — O(log n) insert and O(log n) search. The problem's constraint of increasing timestamps is the key gift.

---

### Q5: What are the edge cases to watch for?

**Answer:** (1) Key doesn't exist — return "". (2) Query timestamp before all stored timestamps — `bisect_right` returns 0, `idx = -1 < 0`, return "". (3) Exact timestamp match — `bisect_right` puts exact matches before the insertion point for `(ts, chr(127))`, so `idx` points to the exact match correctly. (4) Multiple values at different timestamps for same key — sorted order ensures we return the most recent ≤ query.

## 📚 Key Terminology

| Term | Definition |
|------|------------|
| **Floor Query** | Finding the largest key ≤ a given value — the "what was the state at time T?" query pattern |
| **bisect_right** | Python stdlib: returns rightmost insertion index in a sorted array — all elements left are ≤ target |
| **bisect_left** | Returns leftmost insertion index — all elements left are < target (not ≤) |
| **Time Series** | A sequence of data points indexed in time order — the structure underlying all monitoring, metrics, APM data |
| **Append-Only** | Data structure where new entries are only added to the end — guarantees sorted order when values are monotonically increasing |
| **chr(127)** | The DEL character — used as a sentinel to ensure tuples `(ts, chr(127))` sort after all `(ts, value)` pairs with the same timestamp |
| **defaultdict** | Python dict subclass that provides a default value for missing keys — `defaultdict(list)` initializes missing keys with `[]` |
| **Sorted Container** | Data structure that maintains sorted order on insert — `SortedList` from sortedcontainers is the non-stdlib option |

## 💼 The Citi Narrative

**Context:** Citi's APM infrastructure stored per-server metric snapshots every 60 seconds. Post-incident analysis required "what was the CPU for server SRV-1042 at 14:37:00?" — a floor query on the sorted timestamp list for that server.

**Scenario:** The monitoring database had 6,000 servers × 1,440 minutes = 8.64 million timestamp-value pairs per day. Ad-hoc point-in-time queries for specific servers during incident investigation needed to return in under 1 second.

**How this pattern applied:** Each server's readings were stored chronologically — automatically sorted by timestamp. A binary search on the server's timestamp list (bisect_right) returned the floor value in O(log 1440) ≈ 11 comparisons. This is exactly the TimeMap.get() operation.

**Impact:** Incident triage queries that previously required a `WHERE timestamp <= '14:37' ORDER BY timestamp DESC LIMIT 1` full-table-scan on 8.64M rows ran in microseconds against the in-memory sorted structure. This was critical for the 5-minute SLA on incident reports — investigators got exact point-in-time readings instantly.

In [ ]:
# -------------------------------------------------------
# PRACTICE: Implement a simplified version using manual
# binary search (without the bisect module).
# Find the rightmost timestamp <= query_time in the list.
# -------------------------------------------------------

from collections import defaultdict

class TimeMapManual:
    def __init__(self):
        self.store = defaultdict(list)

    def set(self, key, value, timestamp):
        self.store[key].append((timestamp, value))

    def get(self, key, timestamp):
        # Implement binary search manually — do not use bisect
        # Find the rightmost (ts, val) where ts <= timestamp
        pass


# Test
tm = TimeMapManual()
tm.set("foo", "bar", 1)
tm.set("foo", "bar2", 4)
print(tm.get("foo", 4))   # Expected: "bar2"
print(tm.get("foo", 3))   # Expected: "bar"
print(tm.get("foo", 0))   # Expected: ""

## 🎯 Summary: Key Takeaways

### The Pattern
**Binary Search on Sorted Timestamps** — Exploit monotonically increasing timestamps for O(log n) floor queries

### When to Use It
- ✅ Time-series data with monotonically increasing timestamps
- ✅ "Most recent value at or before time T" queries
- ✅ Version control, audit logs, sensor readings, monitoring data
- ❌ **Don't use when:** Timestamps are not guaranteed sorted — need SortedList or sort on insert

### Complexity
| Approach | Time (get) | Space |
|----------|-----------|-------|
| Brute Force (linear) | $O(n)$ | $O(n)$ |
| Optimal (Binary Search) | $O(\log n)$ | $O(n)$ |

### Interview Confidence Checklist
- [ ] Can explain why the "increasing timestamp" constraint enables binary search
- [ ] Can explain bisect_right and the chr(127) sentinel trick
- [ ] Can write the solution from memory in 5 minutes
- [ ] Can explain floor query concept and where it appears in real systems
- [ ] Can connect to monitoring/APM use case with specific numbers

---

**"Simplicity and clarity is Gold."** — Sean's Study Mantra

Master Time-Based KV Store and you've solved the core algorithmic problem behind every monitoring and time-series system. 🚀